## &nbsp; Almost Sure Convergence

*Let $(\Omega, \mathscr{A}, \mathbb{P})$ be a probability space. Let $(X_n)_{n \in \mathbb{N}}$ be a sequence of real random variables from $\Omega$ to $\mathbb{R}$ and let $X : \Omega \to \mathbb{R}$ be a real random variable. We say that $(X_n)_{n \in \mathbb{N}}$ converges to $X$ <u>**almost surely**</u> (or almost everywhere), and we write*

$$X_n \xrightarrow[n \to +\infty]{\text{a.s.}} X,$$

*if $(X_n(\omega))_{n \in \mathbb{N}}$ converges to $X(\omega)$ in $\mathbb{R}$ for almost every $\omega \in \Omega$, that is*

$$\exists A \in \mathscr{A} \text{ such that } \mathbb{P}(A^c) = 0 \text{ and } \forall \omega \in A,\ X_n(\omega) \underset{n \to +\infty}{\longrightarrow} X(\omega).$$


**Theorem** (Strong Law of Large Numbers). *Let $(\Omega, \mathscr{A}, \mathbb{P})$ be a probability space. Let $(X_n)_{n \in \mathbb{N}^*}$ be a sequence of independent and identically distributed real random variables from $\Omega$ to $\mathbb{R}$. Suppose that*
$$X_1 \in L^1_{\mathbb{R}}(\Omega, \mathscr{A}, \mathbb{P}).$$
*Then*
$$\boxed{\frac{1}{n} \sum_{i=1}^{n} X_i \xrightarrow[n \to +\infty]{\text{a.s.}} \mathbb{E}[X_1].}$$

#### Why can we Simulate Option Prices?

Black-Scholes Argument Produces the "Correct" Arbitrage Free Price Implied by the Fundamental Theorem of Asset Pricing (Feynman-Kac)**

$$ V_0 = e^{-rT} \, \mathbb{E}^{\mathbb{Q}}\left[\,\Phi(S_T)\,\right] $$

The Law of Large Numbers (LLN) Ensures Empirical Statistics Converge to Theoretical Ones if the Underlying Distribution is Stable**

 $$
 \frac{1}{N} \sum_{i=1}^N X_i \xrightarrow[]{N \to \infty} \mathbb{E}[X] \quad \text{(Law of Large Numbers)}
 $$


**Following from the above we can:**

1.) Select a stochastic model for the underlying

2.) Discretize and simulate the process to terminal time $T$

3.) Compute the payoff of that sample path $\Phi(S_T)$

4.) Discount it back to the present  $\text{Price} = e^{-rT} \cdot \Phi(S_T)$ and add the value to a list

5.) Repeat at (1) for many iterations and observe convergence by the Law of Large Numbers (LLN)


In [11]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm

# Parameters
S0    = 100
K     = 100
r     = 0.05
sigma = 0.2
T     = 1.0
n_paths = 200
n_steps = 252
dt    = T / n_steps

np.random.seed(42)

# Under risk-neutral measure Q, log-returns are normally distributed:
# ln(S_{t+dt}/S_t) = (r - sigma^2/2)*dt + sigma*sqrt(dt)*Z, Z ~ N(0,1)
# The drift correction -sigma^2/2 comes from Ito's lemma applied to ln(S_t)
Z          = np.random.normal(0, 1, (n_paths, n_steps))
increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

# S_t = S0 * exp(sum of increments up to t)
# cumsum accumulates log-returns along time axis (axis=1)
# prepend zeros so that column 0 corresponds to t=0: S0 * exp(0) = S0
log_paths = np.cumsum(increments, axis=1)
S_paths   = S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), log_paths]))

# S_T ~ LogNormal(ln(S0) + (r - sigma^2/2)*T, sigma^2*T)
# European call payoff: max(S_T - K, 0)
# Price = e^{-rT} * E^Q[max(S_T - K, 0)] approximated by sample mean (LLN)
S_T     = S_paths[:, -1]
payoffs = np.maximum(S_T - K, 0)
C_MC    = np.exp(-r * T) * np.mean(payoffs)

# Closed-form Black-Scholes price for verification
# C_BS = S0*N(d1) - K*e^{-rT}*N(d2)
# d1 = (ln(S0/K) + (r + sigma^2/2)*T) / (sigma*sqrt(T))
# d2 = d1 - sigma*sqrt(T)
d1   = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
d2   = d1 - sigma * np.sqrt(T)
C_BS = S0 * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

time_grid = np.linspace(0, T, n_steps + 1)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"GBM Paths (S0={S0}, σ={sigma}, r={r})",
        f"S_T Distribution — Strike K={K}"
    ),
    column_widths=[0.6, 0.4],
    horizontal_spacing=0.1
)

# Green: S_T > K (in the money, positive payoff)
# Red:   S_T < K (out of the money, payoff = 0)
for i in range(n_paths):
    color = 'rgba(0,200,100,0.15)' if S_T[i] > K else 'rgba(255,60,60,0.15)'
    fig.add_trace(go.Scatter(
        x=time_grid, y=S_paths[i],
        mode='lines',
        line=dict(width=0.8, color=color),
        showlegend=False
    ), row=1, col=1)

fig.add_hline(y=K, line=dict(color='white', dash='dash', width=1.5),
              annotation_text=f"Strike K={K}", row=1, col=1)

# S_T follows a log-normal distribution
# green bars: in the money (S_T > K), red bars: out of the money
hist_y, bin_edges = np.histogram(S_T, bins=50, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bar_colors  = ['rgba(0,200,100,0.6)' if b > K else 'rgba(255,60,60,0.6)'
               for b in bin_centers]

fig.add_trace(go.Bar(
    x=bin_centers, y=hist_y,
    width=(bin_edges[1] - bin_edges[0]),
    marker_color=bar_colors,
    showlegend=False
), row=1, col=2)

fig.add_vline(x=K, line=dict(color='white', dash='dash', width=1.5),
              row=1, col=2)

fig.update_layout(
    title=dict(
        text=(f"European Call Option — "
              f"Monte Carlo: <b>{C_MC:.4f}</b> | "
              f"Black-Scholes: <b>{C_BS:.4f}</b>"),
        font=dict(size=14)
    ),
    height=500,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white', size=11),
)

fig.update_xaxes(title_text='Time (t)', row=1, col=1,
                 showgrid=True, gridcolor='rgba(140,140,140,0.2)')
fig.update_yaxes(title_text='S_t', row=1, col=1,
                 showgrid=True, gridcolor='rgba(140,140,140,0.2)')
fig.update_xaxes(title_text='S_T', row=1, col=2,
                 showgrid=True, gridcolor='rgba(140,140,140,0.2)')
fig.update_yaxes(title_text='Density', row=1, col=2,
                 showgrid=True, gridcolor='rgba(140,140,140,0.2)')

fig.show()